# Main Functions

##Imports

In [1]:
from scipy.optimize import newton
import numpy as np

## Functions definition

In [2]:
def calculate_energy_requirement_per_mission(duration_takeoff, duration_climb,
    duration_cruise, duration_descent, duration_landing):
    """
    Calculates the total energy requirement for an eVTOL mission.
    Returns:
    - Total energy requirement for the mission (kWh).
    """
    c_D = c_D0 + C_L**2/(np.pi * AR_wing * oswald_number)
    Drag = (1/2) * air_density * cruise_speed * wet_area * c_D
    power_takeoff = Thrust**(3/2)/(figure_merit*(2*air_density*rotor_disc_area)**(1/2))
    power_climb = power_takeoff * (vertical_velocity_climb/(2*induced_velocity_hover) + ((vertical_velocity_climb/(2*induced_velocity_hover))**2 + 1)**(1/2))
    power_cruise = 1/propeller_efficiency * Drag * cruise_speed
    power_descent = power_takeoff
    power_landing = power_takeoff

    print(f'Power for takeoff: {power_takeoff:.2f} W')

    # Calculate energy consumed in each phase
    energy_takeoff = power_takeoff * duration_takeoff
    energy_climb = power_climb * duration_climb
    energy_cruise = power_cruise * duration_cruise
    energy_descent = power_descent * duration_descent
    energy_landing = power_landing * duration_landing

    print(f'Energy requirement for takeoff: {energy_takeoff:.2f} Wh')

    # Sum the energy for all phases to get the total mission energy requirement
    total_energy_requirement = (energy_takeoff + energy_climb + energy_cruise +
                                energy_descent + energy_landing)
    print(f'Energy requirement total: {total_energy_requirement:.2f} Wh')

    return total_energy_requirement, power_takeoff, power_climb, power_cruise, power_descent, power_landing

def energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency):
    return total_energy_requirement *(1 + SoCmin)/ (specific_energy_density * system_efficiency)

def airframe_mass(mtow):
    # Placeholder for actual equation from the paper
    mass_fuselage = 14.86*mtow**0.144 * (fuselage_length/maximum_fuselage_section_perimeter)**0.778 * fuselage_length**0.383 * number_ocupants**0.455
    mass_wing = 0.04674*mtow**0.397 * wing_area**0.360 * wing_efficiency**0.397 * AR_wing**1.712
    mass_horizontal_tail = ((3.184*mtow**0.887 * horizontal_tail_area**0.101 * AR_horizontal_tail**0.101)/(174.04* root_chord_horizontal_tail**0.223))
    mass_vertical_tail = ((1.68*mtow**0.567 * vertical_tail_area**1.249 * AR_vertical_tail**0.482)/(639.95*root_chord_vertical_tail**0.747 * (np.cos(vertical_tail_quarter_chord_seet_angle))**0.882))
    mass_landing_gear = 0.054 * landing_gear_length**0.501*(mtow*landing_gear_efficiency)**0.684
    return mass_fuselage + mass_wing + mass_horizontal_tail + mass_vertical_tail + mass_landing_gear

def propulsion_system_mass(mtow, power_takeoff_requirement):
    mass_motors = power_takeoff_requirement/(5920*0.95) # EM mass eq: 5.92 kW/kg from Finger
    mass_propellers = 0.144 * (propeller_diameter * power_takeoff_requirement/(1000) * number_blades_propeller**0.5)**0.782
    return mass_motors + mass_propellers

# Function to be solved using Newton-Raphson, representing the difference between MTOW and the sum of its components
def mtow_balance(mtow, payload, specific_energy_density, system_efficiency):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass = energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (mass_payload + es_mass + af_mass + ps_mass)


## Inputs
Please add the inputs considering your initial estimations for the configuration, payload mass and number of motors. DO NOT edit this cell. Below this one you will find the configuration analysis for each design.

In [30]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 30/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 2000  #kg

c_D0 = 0.25
C_L = 0.6
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 100 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 10 #m2
vertical_velocity_climb = 3 #m/s
induced_velocity_hover = 3 #m/s
propeller_efficiency = 0.65
AR_wing = 10

# Constants and initial parameters
mass_payload = 00  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 5
maximum_fuselage_section_perimeter = 5
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2 #m
number_blades_propeller = 3 #m



## Full-electric consideration

In [31]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print('mass_batteries', energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency))
print('mass_estructure', airframe_mass(mtow_estimated))
print('mass_propulsion', propulsion_system_mass(mtow_estimated, power_takeoff))
print('mass_payload', mass_payload)
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
mass_batteries 1055.9708633860014
mass_estructure 282.1419249653256
mass_propulsion 291.41999223584685
mass_payload 0
Estimated MTOW: 1629.53 kg


## Hybrid consideration

In [5]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.1  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print('mass_battery', battery_mass)
print('mass_fuel', fuel_mass)
print('mass_turbine', turbine_mass)
print('mass_estructure', airframe_mass(mtow_estimated))
print('mass_propulsion', propulsion_system_mass(mtow_estimated, power_takeoff))
print('mass_payload', mass_payload)
print(f"Estimated MTOW with Hybrid System: {mtow_estimated:.2f} kg")

Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
Power for takeoff: 1120233.31 W
Energy requirement for takeoff: 18670.56 Wh
Energy requirement total: 153995.75 Wh
hybridization_level 10.0 %
mass_battery 950.3737770474013
mass_fuel 6.416489621269106
mass_turbine 33.84390661730635
mass_estructure 305.4944417721419
mass_propulsion 291.41999223584685
mass_payload 500
Estimated MTOW with Hybrid System: 2087.55 kg


# Configuration Analysis

## Vectored Thrust Canards

In [6]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 30/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.25
C_L = 0.6
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 100 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 10 #m2
vertical_velocity_climb = 3 #m/s
induced_velocity_hover = 3 #m/s
propeller_efficiency = 0.65
AR_wing = 10

# Constants and initial parameters
mass_payload = 500  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 5
maximum_fuselage_section_perimeter = 5
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 3

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [7]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print('mass_batteries', energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency))
print('mass_estructure', airframe_mass(mtow_estimated))
print('mass_propulsion', propulsion_system_mass(mtow_estimated, power_takeoff))
print('mass_payload', mass_payload)
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
mass_batteries 1917.2858117377602
mass_estructure 357.42243714674566
mass_propulsion 538.5999333729471
mass_payload 500
Estimated MTOW: 3313.31 kg


In [8]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.9  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print('mass_battery', battery_mass)
print('mass_fuel', fuel_mass)
print('mass_turbine', turbine_mass)
print('mass_estructure', airframe_mass(mtow_estimated))
print('mass_propulsion', propulsion_system_mass(mtow_estimated, power_takeoff))
print('mass_payload', mass_payload)
print(f"Estimated MTOW with Hybrid System: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
hybridization_level 90.0 %
mass_battery 191.72858117377598
mass_fuel 104.85156782940878
mass_turbine 587.7877689478929
mass_estructure 312.45389926162136
mass_propulsion 538.5999333729471
mass_payload 500
Estimated MTOW with Hybrid System: 2235.42 kg


## Vectored Thrust Booms

In [9]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 30/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.25
C_L = 0.6
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 100 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 10 #m2
vertical_velocity_climb = 3 #m/s
induced_velocity_hover = 3 #m/s
propeller_efficiency = 0.65
AR_wing = 10

# Constants and initial parameters
mass_payload = 500  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 5
maximum_fuselage_section_perimeter = 5
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 3

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [10]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print('mass_batteries', energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency))
print('mass_estructure', airframe_mass(mtow_estimated))
print('mass_propulsion', propulsion_system_mass(mtow_estimated, power_takeoff))
print('mass_payload', mass_payload)
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
mass_batteries 1917.2858117377602
mass_estructure 357.42243714674566
mass_propulsion 538.5999333729471
mass_payload 500
Estimated MTOW: 3313.31 kg


In [11]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.1  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print('mass_battery', battery_mass)
print('mass_fuel', fuel_mass)
print('mass_turbine', turbine_mass)
print('mass_estructure', airframe_mass(mtow_estimated))
print('mass_propulsion', propulsion_system_mass(mtow_estimated, power_takeoff))
print('mass_payload', mass_payload)
print(f"Estimated MTOW with Hybrid System: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 279604.18 Wh
hybridization_level 10.0 %
mass_battery 1725.5572305639841
mass_fuel 11.65017420326764
mass_turbine 65.30975210532144
mass_estructure 352.8397692136366
mass_propulsion 538.5999333729471
mass_payload 500
Estimated MTOW with Hybrid System: 3193.96 kg


## L+C with fixed open rotors

In [12]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 60/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.25
C_L = 0.4
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 50 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 10 #m2
vertical_velocity_climb = 2 #m/s
induced_velocity_hover = 2 #m/s
propeller_efficiency = 0.65
AR_wing = 12

# Constants and initial parameters
mass_payload = 400  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 4
maximum_fuselage_section_perimeter = 4
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 2.5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 2.5
AR_vertical_tail = 10
root_chord_vertical_tail = 1
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 3

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [13]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print(f'mass_batteries: {energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency):.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
mass_batteries: 1849.08 kg
mass_estructure: 371.74 kg
mass_propulsion: 538.60 kg
mass_payload: 400.00 kg
Estimated MTOW: 3159.42 kg


In [14]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.5  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print(f'mass_battery: {battery_mass:.2f} kg')
print(f'mass_fuel: {fuel_mass:.2f} kg')
print(f'mass_turbine: {turbine_mass:.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 269657.51 Wh
hybridization_level 50.0 %
mass_battery: 924.54 kg
mass_fuel: 56.18 kg
mass_turbine: 326.55 kg
mass_estructure: 346.86 kg
mass_propulsion: 538.60 kg
mass_payload: 400.00 kg
Estimated MTOW: 2592.73 kg


## L+C with fixed ducted rotors

In [15]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 30/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.05
C_L = 0.7
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 100 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 8 #m2
vertical_velocity_climb = 2 #m/s
induced_velocity_hover = 2 #m/s
propeller_efficiency = 0.85
AR_wing = 10

# Constants and initial parameters
mass_payload = 400  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 4
maximum_fuselage_section_perimeter = 4
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 3

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [16]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print(f'mass_batteries: {energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency):.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
mass_batteries: 2028.21 kg
mass_estructure: 346.01 kg
mass_propulsion: 598.03 kg
mass_payload: 400.00 kg
Estimated MTOW: 3372.25 kg


In [17]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_mass, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.5  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_mass, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print(f'mass_battery: {battery_mass:.2f} kg')
print(f'mass_fuel: {fuel_mass:.2f} kg')
print(f'mass_turbine: {turbine_mass:.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
hybridization_level 50.0 %
mass_battery: 1014.11 kg
mass_fuel: 61.62 kg
mass_turbine: 365.09 kg
mass_estructure: 322.21 kg
mass_propulsion: 598.03 kg
mass_payload: 400.00 kg
Estimated MTOW: 2761.06 kg


# Radim's Configuration Analysis

## eVTOL

In [18]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 45/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.05
C_L = 0.4
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 50 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 10 #m2
vertical_velocity_climb = 1 #m/s
induced_velocity_hover = 2 #m/s
propeller_efficiency = 0.85
AR_wing = 8

# Constants and initial parameters
mass_payload = 400  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 4
maximum_fuselage_section_perimeter = 4
number_ocupants = 4

wing_area = 10
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 4

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [19]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
stall_speed = np.sqrt(2*mtow_estimated/(air_density*wing_area*C_L))

print(f'mass_batteries: {energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency):.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")
print(f'stall_speed: {stall_speed} m/s')

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
mass_batteries: 1629.62 kg
mass_estructure: 279.58 kg
mass_propulsion: 556.96 kg
mass_payload: 400.00 kg
Estimated MTOW: 2866.16 kg
stall_speed: 34.55768790726269 m/s


In [20]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_mass, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.5  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_mass, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print(f'mass_battery: {battery_mass:.2f} kg')
print(f'mass_fuel: {fuel_mass:.2f} kg')
print(f'mass_turbine: {turbine_mass:.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 36029.21 Wh
Energy requirement total: 237653.01 Wh
hybridization_level 50.0 %
mass_battery: 814.81 kg
mass_fuel: 49.51 kg
mass_turbine: 326.55 kg
mass_estructure: 263.65 kg
mass_propulsion: 556.96 kg
mass_payload: 400.00 kg
Estimated MTOW: 2411.48 kg


## eSTOL

In [21]:
!clear

duration_takeoff = 0/60  # 1 minutes in hours
duration_climb = 1/60  # 6 minutes in hours
duration_cruise = 60/60  # 30 minutes in hours
duration_descent = 1/60 # 6 minutes in hours
duration_landing = 0/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.025
C_L = 0.6
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 75 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 10 #m2
vertical_velocity_climb = 1 #m/s
induced_velocity_hover = 2 #m/s
propeller_efficiency = 0.85
AR_wing = 10

# Constants and initial parameters
mass_payload = 400  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 4
maximum_fuselage_section_perimeter = 4
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 2

propeller_diameter = 2
number_blades_propeller = 4

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [22]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
stall_speed = np.sqrt(2*mtow_estimated/(air_density*wing_area*C_L))

print(f'mass_batteries: {energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency):.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")
print(f'stall_speed: {stall_speed} m/s')

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
mass_batteries: 583.05 kg
mass_estructure: 279.72 kg
mass_propulsion: 556.96 kg
mass_payload: 400.00 kg
Estimated MTOW: 1819.74 kg
stall_speed: 18.357237153258556 m/s


In [23]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_mass, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.5  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_mass, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print(f'mass_battery: {battery_mass:.2f} kg')
print(f'mass_fuel: {fuel_mass:.2f} kg')
print(f'mass_turbine: {turbine_mass:.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
Power for takeoff: 2161752.79 W
Energy requirement for takeoff: 0.00 Wh
Energy requirement total: 85028.54 Wh
hybridization_level 50.0 %
mass_battery: 291.53 kg
mass_fuel: 17.71 kg
mass_turbine: 326.55 kg
mass_estructure: 282.50 kg
mass_propulsion: 556.96 kg
mass_payload: 400.00 kg
Estimated MTOW: 1875.25 kg


## STOL

In [24]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 30/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.05
C_L = 0.7
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 100 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 8 #m2
vertical_velocity_climb = 2 #m/s
induced_velocity_hover = 2 #m/s
propeller_efficiency = 0.85
AR_wing = 10

# Constants and initial parameters
mass_payload = 400  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 4
maximum_fuselage_section_perimeter = 4
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 3

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [25]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print(f'mass_batteries: {energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency):.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
mass_batteries: 2028.21 kg
mass_estructure: 346.01 kg
mass_propulsion: 598.03 kg
mass_payload: 400.00 kg
Estimated MTOW: 3372.25 kg


In [26]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_mass, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.5  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_mass, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print(f'mass_battery: {battery_mass:.2f} kg')
print(f'mass_fuel: {fuel_mass:.2f} kg')
print(f'mass_turbine: {turbine_mass:.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
hybridization_level 50.0 %
mass_battery: 1014.11 kg
mass_fuel: 61.62 kg
mass_turbine: 365.09 kg
mass_estructure: 322.21 kg
mass_propulsion: 598.03 kg
mass_payload: 400.00 kg
Estimated MTOW: 2761.06 kg


## CTOL

In [27]:
!clear

duration_takeoff = 1/60  # 1 minutes in hours
duration_climb = 2/60  # 6 minutes in hours
duration_cruise = 30/60  # 30 minutes in hours
duration_descent = 2/60 # 6 minutes in hours
duration_landing = 1/60  # 5 minutes in hours

# Initial guess for MTOW
mtow_guess = 3100  # kg

c_D0 = 0.05
C_L = 0.7
oswald_number = 0.5
air_density = 1.2 #kg/m3
cruise_speed = 100 #m/s
wet_area = 15 #m2
Thrust = mtow_guess*9.8 #N
figure_merit = 0.5
rotor_disc_area = 8 #m2
vertical_velocity_climb = 2 #m/s
induced_velocity_hover = 2 #m/s
propeller_efficiency = 0.85
AR_wing = 10

# Constants and initial parameters
mass_payload = 400  # kg

#energy_requirement_per_mission = total_energy_mission  # Wh, calculated in the other cell
battery_specific_energy_density = 250  # Wh/kg
battery_system_efficiency = 0.7  # assumed efficiency
SoCmin = 0.2 # minimum battery state of charge

fuselage_length = 4
maximum_fuselage_section_perimeter = 4
number_ocupants = 4

wing_area = 15
wing_efficiency = 0.5

horizontal_tail_area = 5
AR_horizontal_tail = 10
root_chord_horizontal_tail = 1

vertical_tail_area = 5
AR_vertical_tail = 10
root_chord_vertical_tail = 0.5
vertical_tail_quarter_chord_seet_angle = 15*np.pi/180

landing_gear_length = 3 #m
landing_gear_efficiency = 1 #m

power_margin = 0.2
number_motors = 8

propeller_diameter = 2
number_blades_propeller = 3

# Initial guess for MTOW
mtow_guess = 2000  # kg

In [28]:
# Solve for MTOW using the Newton-Raphson method
mtow_estimated = newton(mtow_balance, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency))
total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)

print(f'mass_batteries: {energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency):.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
mass_batteries: 2028.21 kg
mass_estructure: 346.01 kg
mass_propulsion: 598.03 kg
mass_payload: 400.00 kg
Estimated MTOW: 3372.25 kg


In [29]:
!clear
from scipy.optimize import newton
import numpy as np

# Updated energy system mass calculation to include hybrid configuration
def hybrid_energy_system_mass(total_energy_requirement, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level):
    battery_mass = (total_energy_requirement * (1 + SoCmin)) / (specific_energy_density * system_efficiency)*(1 - hybridization_level)
    # Assuming fuel has a specific energy of 12 kWh/kg (approx. for Jet-A fuel) and turbine_efficiency
    fuel_mass = total_energy_requirement * hybridization_level / (fuel_specific_energy * 1000 * turbine_efficiency)
    turbine_mass = power_takeoff * hybridization_level/3310  # P*gt = 3.31kW/kg by Finger.
    return battery_mass + turbine_mass + fuel_mass, battery_mass, fuel_mass, turbine_mass

# Adjusted function for the mass balance that includes the hybrid system
def mtow_balance_hybrid(mtow, payload, specific_energy_density, system_efficiency, turbine_mass, turbine_efficiency, hybridization_level):
    total_energy_mission, power_takeoff, power_climb, power_cruise, power_descent, power_landing = calculate_energy_requirement_per_mission(duration_takeoff,
                                                                duration_climb,
                                                                duration_cruise,
                                                                duration_descent,
                                                                duration_landing)
    es_mass, battery_mass, fuel_mass, turbine = hybrid_energy_system_mass(total_energy_mission, specific_energy_density, system_efficiency, power_takeoff, turbine_efficiency, hybridization_level)
    af_mass = airframe_mass(mtow)
    ps_mass = propulsion_system_mass(mtow, power_takeoff)
    return mtow - (payload + es_mass + af_mass + ps_mass)

# Example inputs for the hybrid system
turbine_efficiency = 0.2  # Example efficiency of converting fuel to electricity
fuel_specific_energy = 12 # kWh/kg
hybridization = 0.5  # Hybridization level. Percentage of energy from the turbine. More than 0.01 (so it makes sense)

# Adjust the initial call to newton's method for MTOW estimation
mtow_estimated = newton(mtow_balance_hybrid, mtow_guess, args=(mass_payload, battery_specific_energy_density, battery_system_efficiency, turbine_mass, turbine_efficiency, hybridization))

# Additional print statements to show hybrid system masses
_, battery_mass, fuel_mass, turbine_mass = hybrid_energy_system_mass(total_energy_mission, battery_specific_energy_density, battery_system_efficiency, power_takeoff, turbine_efficiency, hybridization)

print('hybridization_level', hybridization*100, '%')
print(f'mass_battery: {battery_mass:.2f} kg')
print(f'mass_fuel: {fuel_mass:.2f} kg')
print(f'mass_turbine: {turbine_mass:.2f} kg')
print(f'mass_estructure: {airframe_mass(mtow_estimated):.2f} kg')
print(f'mass_propulsion: {propulsion_system_mass(mtow_estimated, power_takeoff):.2f} kg')
print(f'mass_payload: {mass_payload:.2f} kg')
print(f"Estimated MTOW: {mtow_estimated:.2f} kg")

Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
Power for takeoff: 2416913.10 W
Energy requirement for takeoff: 40281.88 Wh
Energy requirement total: 295780.98 Wh
hybridization_level 50.0 %
mass_battery: 1014.11 kg
mass_fuel: 61.62 kg
mass_turbine: 365.09 kg
mass_estructure: 322.21 kg
mass_propulsion: 598.03 kg
mass_payload: 400.00 kg
Estimated MTOW: 2761.06 kg
